In [1]:
from promptsmith.dspy_init import get_dspy
from promptsmith.tasks.bad_to_good.bulletize_text_2 import BulletizeText
from promptsmith.judges.judge_bullet_structure import JudgeBulletStructure
from promptsmith.judges.judge_coverage import JudgeCoverage
from promptsmith.judges.judge_focus_relevance import JudgeFocusRelevance
from promptsmith.judges.judge_redundancy import JudgeRedundancy
from promptsmith.refiners import Refiner
from promptsmith.evaluation.task_evaluator import TaskEvaluator
from promptsmith.utils.display import display_evaluation_result
from promptsmith.refiners.orchestrator import RefinementOrchestrator

dspy, lm = get_dspy()

/Users/yanivgal/dev/ai21/promptsmith/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
with open('../../data/text_01.txt', 'r') as f:
# with open('../../data/text_02_long_text.txt', 'r') as f:
# with open('../../data/text_03.txt', 'r') as f:
    text = f.read()

#### original text

In [3]:
print(text)

The first question in this study stated: What is the strength and direction of the relationship between faculty's perceptions of the importance of course design, communication, time management, and technical competency, and their ability to teach online? To address this question, descriptive statistics (mean and standard deviation) by individual competency items of each of the four competency dimensions: course design, course communication, time management, and technical competency,  were reported in Table 2 on page 23. Most of the items in each competency were rated high for both constructs, Importance, and Ability.
The items manage grades online (M = 4.73) and creating online assignments (M = 4.68)

were rated the highest in the course design competency. In the course communication competency, responding to student questions promptly (M = 4.79) and providing feedback on assignments (M = 4.65) were rated the highest. In time management, scheduling time to design the course prior to de

#### refining

In [4]:
bulletize_text_evaluator = TaskEvaluator(
    task=dspy.ChainOfThought(BulletizeText),
    judges={
        'structure': dspy.Predict(JudgeBulletStructure),
        'coverage': dspy.Predict(JudgeCoverage),
        'focus_relevance': dspy.Predict(JudgeFocusRelevance),
        'redundancy': dspy.Predict(JudgeRedundancy)
    },
    weights={
        'structure': 0.5,
        'coverage': 0.2,
        'focus_relevance': 0.15,
        'redundancy': 0.15,
    }
)

In [5]:
orchestrator = RefinementOrchestrator(
    evaluator=bulletize_text_evaluator,
    refiner=dspy.Predict(Refiner),
    max_iterations=5,
    score_threshold=0.96
)

In [6]:
orchestrator.refine(text)


🚀 Starting refinement process (max 5 iterations)

🔄 Iteration 1/5
----------------------------------------
🔍 Evaluating current output...

Combined score: 0.76

structure: 0.65
- The title is appropriate and meets the length requirement.
- The one-line summary is missing; it should be included immediately after the title.
- Each section begins with a meaningful H2 heading, which is correct.
- The bullet points are mostly well-structured, but the use of bold labels is inconsistent; some sections have them while others do not.
- Each section has a closing line, but they do not follow the required format of being italicized.
- The sections are properly separated by three dashes.
- There is no extraneous text present.

Overall, the output has several structural issues, particularly with the missing one-line summary, inconsistent use of bold labels, and the format of the closing lines.

coverage: 0.90
The output text effectively captures the key ideas from the input text. It maintains the 

{'output': "# Faculty Perceptions of Online Teaching Competencies\n\nThis document explores faculty perceptions of online teaching competencies and their teaching abilities.\n\n## Relationship Between Perceptions and Ability\n- **Mean Ratings:**\n  - **Course Design:** Manage grades online (M = 4.73), Creating online assignments (M = 4.68)\n  - **Course Communication:** Responding to student questions promptly (M = 4.79), Providing feedback on assignments (M = 4.65)\n  - **Time Management:** Scheduling time to design the course (M = 4.65), Spending weekly hours to grade assignments (M = 4.54)\n  - **Technical Competency:** Basic computer operations (M = 4.74), Navigating the learning management system (M = 4.67)\n\n- **Overall Insight:** There is a weak positive relationship between the perceived importance of these competencies and faculty's ability to teach online (r(223) = .249, p < .001), indicating a need for more support in course design and communication.\n\n*This section highli

In [7]:
print(orchestrator.history[0].output)


# Faculty Perceptions of Online Teaching Competencies

## Research Question 1: Relationship Between Perceptions and Ability
- **Question:** What is the strength and direction of the relationship between faculty's perceptions of the importance of course design, communication, time management, and technical competency, and their ability to teach online?
- Descriptive statistics (mean and standard deviation) reported in Table 2 on page 23.
- High ratings for both constructs, Importance and Ability, across competencies.
  - **Course Design:** 
    - Manage grades online (M = 4.73)
    - Creating online assignments (M = 4.68)
  - **Course Communication:** 
    - Responding to student questions promptly (M = 4.79)
    - Providing feedback on assignments (M = 4.65)
  - **Time Management:** 
    - Scheduling time to design the course prior to delivery (M = 4.65)
    - Spending weekly hours to grade assignments (M = 4.54)
  - **Technical Competency:** 
    - Complete basic computer operations (

In [8]:
print(orchestrator.history[-1].output)

# Faculty Perceptions of Online Teaching Competencies

This document explores faculty perceptions of online teaching competencies and their teaching abilities.

## Relationship Between Perceptions and Ability
- **Mean Ratings:**
  - **Course Design:** Manage grades online (M = 4.73), Creating online assignments (M = 4.68)
  - **Course Communication:** Responding to student questions promptly (M = 4.79), Providing feedback on assignments (M = 4.65)
  - **Time Management:** Scheduling time to design the course (M = 4.65), Spending weekly hours to grade assignments (M = 4.54)
  - **Technical Competency:** Basic computer operations (M = 4.74), Navigating the learning management system (M = 4.67)

- **Overall Insight:** There is a weak positive relationship between the perceived importance of these competencies and faculty's ability to teach online (r(223) = .249, p < .001), indicating a need for more support in course design and communication.

*This section highlights the relationship bet